# Case Study: Psychometric Analysis with Crossed Random Effects

**Domain:** Psychology / Psychometrics  
**Design:** Crossed Random Effects (Subjects × Items)  
**Analysis:** Mixed Models for Response Time Data

## Background

A cognitive psychology lab studies lexical decision times. Participants (subjects) respond to multiple words (items), deciding if each is a real word or pseudoword.

**Research questions:**
1. Do word frequency and length affect response times?
2. How much variability comes from subjects vs items?
3. Are there subject-item interactions?

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from aurora.models.gamm import fit_gamm

sns.set_style('whitegrid')
np.random.seed(123)

## 1. Data Generation

Simulate crossed design:
- **40 subjects** × **80 items** = 3200 trials
- Fixed effects: word frequency, word length
- Random effects: subject intercepts, item intercepts

In [ ]:
# Parameters
n_subjects = 40
n_items = 80

# True effects
baseline_rt = 600  # ms
frequency_effect = -50  # high freq faster
length_effect = 30  # longer words slower
subject_sd = 80  # between-subject variability
item_sd = 50  # between-item variability
residual_sd = 100  # trial-to-trial noise

# Generate random effects
subject_effects = np.random.randn(n_subjects) * subject_sd
item_effects = np.random.randn(n_items) * item_sd

# Generate items with properties
items = pd.DataFrame({
    'item_id': range(n_items),
    'frequency': np.random.choice([0, 1], n_items),  # 0=low, 1=high
    'length': np.random.randint(3, 10, n_items)  # 3-9 letters
})

# Generate crossed data
data = []
for subj in range(n_subjects):
    for _, item in items.iterrows():
        rt = (baseline_rt +
              subject_effects[subj] +
              item_effects[item['item_id']] +
              frequency_effect * item['frequency'] +
              length_effect * (item['length'] - 6) +  # centered
              np.random.randn() * residual_sd)
        
        data.append({
            'subject': subj,
            'item': item['item_id'],
            'frequency': item['frequency'],
            'length': item['length'],
            'RT': max(200, rt)  # minimum RT
        })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} trials from {n_subjects} subjects × {n_items} items")

## 2. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency effect
df.boxplot(column='RT', by='frequency', ax=axes[0])
axes[0].set_xlabel('Word Frequency (0=Low, 1=High)')
axes[0].set_ylabel('Response Time (ms)')
axes[0].set_title('RT by Word Frequency')
plt.sca(axes[0])
plt.xticks([1, 2], ['Low', 'High'])

# Length effect
axes[1].scatter(df['length'], df['RT'], alpha=0.1, s=10)
length_means = df.groupby('length')['RT'].mean()
axes[1].plot(length_means.index, length_means.values, 'r-', linewidth=3)
axes[1].set_xlabel('Word Length (letters)')
axes[1].set_ylabel('Response Time (ms)')
axes[1].set_title('RT by Word Length')

plt.tight_layout()
plt.show()

## 3. GAMM with Crossed Random Effects

**Model**: RT ~ frequency + length + (1 | subject) + (1 | item)

In [ ]:
# Center length for interpretability
df['length_c'] = df['length'] - df['length'].mean()

result = fit_gamm(
    formula='RT ~ frequency + length_c + (1 | subject) + (1 | item)',
    data=df,
    family='gaussian',
    covariance='identity'
)

print("\nGAMM Results: Crossed Random Effects")
print("="*60)
print(f"Converged: {result.converged}")

# Fixed effects
print("\nFixed Effects:")
print(f"  Intercept: {result.beta_parametric[0]:.2f} ms")
print(f"  Frequency: {result.beta_parametric[1]:.2f} ms (true: {frequency_effect})")
print(f"  Length: {result.beta_parametric[2]:.2f} ms/letter (true: {length_effect})")

# Random effects
subject_sd_est = np.sqrt(result.variance_components[0][0, 0])
item_sd_est = np.sqrt(result.variance_components[1][0, 0])
residual_sd_est = np.sqrt(result.residual_variance)

print("\nVariance Components:")
print(f"  Subject SD: {subject_sd_est:.2f} ms (true: {subject_sd})")
print(f"  Item SD: {item_sd_est:.2f} ms (true: {item_sd})")
print(f"  Residual SD: {residual_sd_est:.2f} ms (true: {residual_sd})")

# Variance decomposition
total_var = result.variance_components[0][0,0] + result.variance_components[1][0,0] + result.residual_variance
subj_pct = 100 * result.variance_components[0][0,0] / total_var
item_pct = 100 * result.variance_components[1][0,0] / total_var
resid_pct = 100 * result.residual_variance / total_var

print("\nVariance Explained:")
print(f"  Subjects: {subj_pct:.1f}%")
print(f"  Items: {item_pct:.1f}%")
print(f"  Residual: {resid_pct:.1f}%")

## 4. Random Effects Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subject random effects
subject_res = [result.random_effects['subject'][i][0] for i in range(n_subjects)]
axes[0].hist(subject_res, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Subject Random Effect (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Subject Variability (SD={subject_sd_est:.1f} ms)')
axes[0].grid(alpha=0.3, axis='y')

# Item random effects
item_res = [result.random_effects['item'][i][0] for i in range(n_items)]
axes[1].hist(item_res, bins=20, edgecolor='black', alpha=0.7, color='darkgreen')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Item Random Effect (ms)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Item Variability (SD={item_sd_est:.1f} ms)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Conclusions

### Key Findings

1. **Frequency Effect**: High-frequency words processed ~50ms faster
2. **Length Effect**: Each additional letter adds ~30ms
3. **Individual Differences**: 
   - Subjects vary by ~80ms (fast vs slow responders)
   - Items vary by ~50ms (easy vs hard words)
4. **Crossed Design**: Properly accounts for both sources of non-independence

### Why Crossed Random Effects Matter

Ignoring either subjects or items leads to:
- **Anti-conservative tests** (inflated Type I error)
- **Incorrect standard errors**
- **Non-generalizable findings**

Crossed random effects allow generalization to:
- New subjects from the population
- New items from the language

### References

- Baayen, Davidson, & Bates (2008). *Mixed-effects modeling with crossed random effects for subjects and items*
- Barr et al. (2013). *Random effects structure for confirmatory hypothesis testing*